Plan: Semantic Annotation Evaluation:
1. Load Saved Annotations — Load the 61 JSON files without making additional Gemini API requests.
2. Shape Analysis — Analyze the distribution of round, oval, irregular, and uncertain shapes.
3. Appearance Analysis — Examine textual characteristics such as anechoic, hypoechoic, and well-defined borders.
4. Image Quality Analysis — Analyze the distribution of image quality ratings: good, moderate, and poor.
5. Uncertainty Analysis — Determine how many images Gemini flagged as uncertain.
6. Qualitative Evaluation — Select 15–20 images for manual assessment of whether the generated descriptions are supported by visible image features.
7. Save Results — Save the evaluation tables and visualizations for the Bachelor thesis.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import json

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/ovarian_ultrasound_vlm"
)

MANIFEST_PATH = PROJECT_DIR / "image_manifest.csv"

PREDICTIONS_DIR = (
    PROJECT_DIR / "predictions" / "main_validation"
)

manifest = pd.read_csv(MANIFEST_PATH)

validation = (
    manifest[manifest["split"] == "validation"]
    .sort_values("case_id")
    .reset_index(drop=True)
)

assert len(validation) == 61

# Load existing predictions without calling Gemini API.
annotations = []

for _, row in validation.iterrows():
    case_id = row["case_id"]
    prediction_path = PREDICTIONS_DIR / f"{case_id}.json"

    assert prediction_path.exists(), (
        f"Missing annotation: {case_id}"
    )

    with open(prediction_path, "r", encoding="utf-8") as f:
        result = json.load(f)

    ann = result["annotation"]

    annotations.append({
        "case_id": case_id,
        "category": row["category"],
        "model": result["model"],
        "follicle_like_count": ann["follicle_like_count"],
        "image_quality": ann["image_quality"],
        "uncertain": ann["uncertain"],
        "description": ann["description"],
        "regions": ann["regions"]
    })

semantic_df = pd.DataFrame(annotations)

assert len(semantic_df) == 61
assert semantic_df["case_id"].is_unique

print("Loaded annotations:", len(semantic_df))
print(
    "Total described regions:",
    semantic_df["follicle_like_count"].sum()
)

display(
    semantic_df[
        [
            "case_id",
            "category",
            "follicle_like_count",
            "image_quality",
            "uncertain",
            "description"
        ]
    ].head(10)
)

 Load the saved Gemini validation annotations for semantic evaluation